# Spark Setup

In [1]:
import glob
import glob
import os
import shutil
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date
)
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

Debug: SparkSession has been created successfully.


Accepting streams

In [2]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


join them with each other so easy process i guess idk ill figure out why later

In [3]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

+---------+--------+-----------+
|camera_id|position|speed_limit|
+---------+--------+-----------+
|        1|   152.5|        110|
|        2|   153.5|        110|
|        3|   154.5|         90|
+---------+--------+-----------+

Debug: Camera loaded: 3 cameras.


In [4]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        os.makedirs(base_path, exist_ok=True)
        temp_path = f"{base_path}/_tmp_batch_{batch_id}"
        (
            batch_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .json(temp_path)
        )
        part_files = glob.glob(f"{temp_path}/part-*.json")
        if part_files:
            target_path = f"{base_path}/res_{batch_id}.json"
            shutil.move(part_files[0], target_path)
        shutil.rmtree(temp_path, ignore_errors=True)
    return _writer

# camera_a_instant_query = (
#     camera_a_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_a")
#     .start()
# )

# camera_b_instant_query = (
#     camera_b_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_b")
#     .start()
# )

# camera_c_instant_query = (
#     camera_c_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_c")
#     .start()
# )

print("Debug: Instantaneous violations have been extracted and combined.")


Debug: Instantaneous violations have been extracted and combined.


average speed violations time baby

In [5]:
# A→B segment join (camera 1 to camera 2)
segment_ab = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source")
        
    )
)

# B→C segment join (camera 2 to camera 3)
segment_bc = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id")
    )
)

# segment_ab = (
#     segment_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment A→B"))
#     .start()
# )

# segment_bc = (
#     segment_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment B→C"))
#     .start()
# )

print("Debug: Segment joins have been defined for A→B and B→C.")

Debug: Segment joins have been defined for A→B and B→C.


In [6]:


def compute_avg_speed(joined_segments):
    return (
        joined_segments
        .withColumn(
            "distance_km",
            spark_abs(col("exit_position") - col("entry_position"))
        )
        .withColumn(
            "travel_time_hours",
            (unix_timestamp(col("exit_time")) - unix_timestamp(col("entry_time"))) / 3600.0
        )
        .withColumn(
            "travel_time_raw",
            unix_timestamp(col("exit_time")) - unix_timestamp(col("entry_time"))
        )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time"
    )
)


average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

ab_avg_violations_query = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_ab"))
    .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/avg_violations/camera_ab")
    .start()
)

bc_avg_violations_query = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_bc"))
    .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/avg_violations/camera_bc")
    .start()
)

# avg_vio_ab_query = (
#     average_violations_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations A→B"))
#     .start()
# )

# avg_vio_bc_query = (
#     average_violations_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations B→C"))
#     .start()
# )


print("Average speed violation detection logic defined.")

Average speed violation detection logic defined.


In [7]:
spark.streams.awaitAnyTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 